In [50]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from pathlib import Path

In [3]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=8

In [6]:
# 1. Load
grav = pd.read_csv(DATA / 'Gravity_csv_V202211' / 'Gravity_V202211.csv', usecols=["iso3_o", "iso3_d", "year", "distw_harmonic", "dist", "contig"], low_memory=False)

In [17]:
# 2. Pick the most recent year (or any reference year)
g = grav[grav.year == 2021].copy()

In [19]:
# 3. Diagnose duplicates (CEPII 有些 ISO3 对应多个内部 country_id)
dups = g.groupby(["iso3_o", "iso3_d"]).size()
print(f"重复的 (origin, destination) 对: {(dups > 1).sum()}")

重复的 (origin, destination) 对: 4293


In [28]:
# 4. Pivot — 用 pivot_table + mean 处理重复
dist_matrix = g.pivot_table(
    index="iso3_o",
    columns="iso3_d",
    values="distw_harmonic",
    aggfunc="mean"
)

In [26]:
# 5. 限制到你的国家列表 + 对齐行列顺序
country_list = sorted(set(g.iso3_o) & set(g.iso3_d))  # 或换成你的 ~190 国家列表
dist_matrix = dist_matrix.reindex(index=country_list, columns=country_list)

In [30]:
# 6. 对角线设为 NaN(避免自我贸易)
np.fill_diagonal(dist_matrix.values, np.nan)

In [31]:
dist_matrix

iso3_d,ABW,AFG,AGO,AIA,ALB,AND,ARE,ARG,ARM,ASM,...,VEN,VGB,VNM,VUT,WLF,WSM,YEM,ZAF,ZMB,ZWE
iso3_o,,,,,,,,,,,,,,,,,,,,,
ABW,NaN,13175.0,9575.0,978.0,9090.0,7570.0,12772.0,5145.0,11109.0,11486.0,...,389.0,873.0,17013.0,13759.0,12052.0,11591.0,12222.0,11317.0,11245.0,11538.0
AFG,13175.0,NaN,7509.0,12212.0,4240.0,5712.0,1598.0,15278.0,2152.0,13758.0,...,13231.0,12303.0,4463.0,12036.0,13198.0,13634.0,3219.0,8185.0,6861.0,7070.0
AGO,9575.0,7509.0,NaN,8973.0,5669.0,5897.0,5910.0,7947.0,6358.0,17309.0,...,9277.0,9138.0,10574.0,15895.0,17243.0,17334.0,4324.0,2507.0,1682.0,2046.0
AIA,978.0,12212.0,8973.0,NaN,8112.0,6592.0,11798.0,5686.0,10134.0,12335.0,...,1064.0,166.0,16231.0,14610.0,12882.0,12435.0,11302.0,10939.0,10656.0,10995.0
ALB,9090.0,4240.0,5669.0,8112.0,NaN,1521.0,3724.0,11628.0,2082.0,16840.0,...,9085.0,8227.0,8704.0,16037.0,16535.0,16759.0,3811.0,7761.0,6255.0,6707.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WSM,11591.0,13634.0,17334.0,12435.0,16759.0,16762.0,14974.0,11069.0,15438.0,125.0,...,11763.0,12275.0,9515.0,2175.0,481.0,NaN,16116.0,14888.0,16165.0,15689.0
YEM,12222.0,3219.0,4324.0,11302.0,3811.0,5145.0,1608.0,12249.0,2846.0,16235.0,...,12085.0,11450.0,6674.0,14028.0,15635.0,16116.0,NaN,5100.0,3681.0,3953.0
ZAF,11317.0,8185.0,2507.0,10939.0,7761.0,8281.0,6635.0,8051.0,7809.0,14886.0,...,10968.0,11096.0,9762.0,13414.0,14741.0,14888.0,5100.0,NaN,1469.0,1049.0


In [32]:
# 8. 检查
print(f"矩阵形状: {dist_matrix.shape}")
print(f"缺失值数量: {dist_matrix.isna().sum().sum()} (含对角线 {len(country_list)})")
print(f"对称性误差 (应为 0): {(dist_matrix - dist_matrix.T).abs().max().max():.6f}")

矩阵形状: (235, 235)
缺失值数量: 235 (含对角线 243)
对称性误差 (应为 0): 0.000000


In [ ]:
dist_matrix.to_csv(RESULTS / 'bilateral-distance-matrix.csv')

In [33]:
country_pseudo_values = pd.read_csv(DATA / 'country-pseudo-values.csv')

In [34]:
country_pseudo_values

,value,country,full name,region,GRIP_region,notes
0,1,IRN,Iran,Middle East,5.0,NaN
1,2,BLM,Saint-Barth??lemy,Latin America and the Caribbean,2.0,NaN
2,3,UGA,Uganda,East Africa,3.0,NaN
3,4,SLE,Sierra Leone,West Africa,3.0,NaN
4,5,VCT,Saint Vincent and the Grenadines,Latin America and the Caribbean,2.0,NaN
...,...,...,...,...,...,...
250,251,MNP,Northern Mariana Islands,Oceania,7.0,NaN
251,252,IOT,British Indian Ocean Territory,Southern Africa,3.0,NaN
252,253,ARM,Armenia,Central and Northern Asia,5.0,Western Asia
253,254,HKG,Hong Kong,Eastern Asia,6.0,NaN


In [38]:
dist_matrix.columns

Index(['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ASM',
       ...
       'VEN', 'VGB', 'VNM', 'VUT', 'WLF', 'WSM', 'YEM', 'ZAF', 'ZMB', 'ZWE'],
      dtype='object', name='iso3_d', length=235)

In [36]:
country_pseudo_values['country'].values

array(['IRN', 'BLM', 'UGA', 'SLE', 'VCT', 'BFA', 'SRB', 'TGO', 'BLZ',
       'MYS', 'SMR', 'ALA', 'VIR', 'BHS', 'SLB', 'MUS', 'ETH', 'POL',
       'MDV', 'AIA', 'TCD', 'HTI', 'SEN', 'NGA', 'TLS', 'CXR', 'EST',
       'PRY', 'AFG', 'LUX', 'WLF', 'GIN', 'ASM', 'ZWE', 'BGR', 'XCL',
       'ESH', 'AND', 'NIU', 'VNM', 'OMN', 'KNA', 'LVA', 'FRO', 'UMI',
       'AUT', 'ISL', 'DOM', 'DMA', 'LIE', 'VGB', 'LBR', 'EGY', 'ATF',
       'XKO', 'AGO', 'IRQ', 'MLI', 'BRA', 'SJM', 'HRV', 'MTQ', 'RWA',
       'LKA', 'ISR', 'BOL', 'MNG', 'MCO', 'LCA', 'WSM', 'NER', 'CMR',
       'TUR', 'QAT', 'TWN', 'BES', 'GNQ', 'CIV', 'TUV', 'NAM', 'KGZ',
       'BEL', 'DNK', 'PLW', 'MKD', 'NPL', 'TTO', 'MSR', 'SUR', 'CZE',
       'GUF', 'STP', 'MDA', 'USA', 'GEO', 'ALB', 'PCN', 'IND', 'MOZ',
       'GLP', 'DEU', 'BVT', 'BIH', 'TZA', 'TUN', 'PAK', 'JOR', 'CYP',
       'LSO', 'ZAF', 'SDN', 'MAF', 'MAR', 'FRA', 'IRL', 'CHN', 'FSM',
       'MEX', 'SXM', 'ARE', 'GBR', 'REU', 'MMR', 'IDN', 'MRT', 'BTN',
       'HUN', 'IMN',

In [39]:
cepii_iso3   = set(dist_matrix.columns)
my_iso3      = set(country_pseudo_values['country'].values)

# 三个集合
both         = cepii_iso3 & my_iso3
only_in_mine = my_iso3 - cepii_iso3   # ← 关键: 我有但 CEPII 没有
only_in_cepii = cepii_iso3 - my_iso3   # CEPII 有但我没用到 (无所谓)

print(f"CEPII: {len(cepii_iso3)}")
print(f"My country_raster: {len(my_iso3)}")
print(f"交集: {len(both)}")
print(f"我有 CEPII 没有的 (需处理): {len(only_in_mine)}")
print(f"   → {sorted(only_in_mine)}")
print(f"CEPII 有我没用到的: {len(only_in_cepii)}")

CEPII: 235
My country_raster: 255
交集: 235
我有 CEPII 没有的 (需处理): 20
   → ['ALA', 'ATA', 'ATF', 'BLM', 'BVT', 'GGY', 'HMD', 'IMN', 'JEY', 'MAF', 'SGS', 'SJM', 'UMI', 'VIR', 'XAD', 'XCL', 'XKO', 'XPI', 'XSP', 'ZNC']
CEPII 有我没用到的: 0


In [40]:
import rasterio as rio

In [42]:
country_raster_1km = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif').read(1)

In [47]:
demand_raster = rio.open(r"F:\aggregates-mining-biodiversity\results\uncertainty-analysis\sand-demand-2023-2050-BAU-50km-M.tif").read(1)

In [49]:
country_pseudo_values

,value,country,full name,region,GRIP_region,notes
0,1,IRN,Iran,Middle East,5.0,NaN
1,2,BLM,Saint-Barth??lemy,Latin America and the Caribbean,2.0,NaN
2,3,UGA,Uganda,East Africa,3.0,NaN
3,4,SLE,Sierra Leone,West Africa,3.0,NaN
4,5,VCT,Saint Vincent and the Grenadines,Latin America and the Caribbean,2.0,NaN
...,...,...,...,...,...,...
250,251,MNP,Northern Mariana Islands,Oceania,7.0,NaN
251,252,IOT,British Indian Ocean Territory,Southern Africa,3.0,NaN
252,253,ARM,Armenia,Central and Northern Asia,5.0,Western Asia
253,254,HKG,Hong Kong,Eastern Asia,6.0,NaN


In [43]:
country_raster_1km

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(16922, 33844), dtype=uint8)

In [51]:
# 1. 加载 50km country raster (与 demand 分辨率对齐)
country_raster_50km = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-50km.tif').read(1)

In [52]:
# 2. 加载三种材料的总需求 (用 BAU-M 作为代表性场景)
sand      = rio.open(r"F:\aggregates-mining-biodiversity\results\uncertainty-analysis\sand-demand-2023-2050-BAU-50km-M.tif").read(1)
gravel    = rio.open(r"F:\aggregates-mining-biodiversity\results\uncertainty-analysis\gravel-demand-2023-2050-BAU-50km-M.tif").read(1)
limestone = rio.open(r"F:\aggregates-mining-biodiversity\results\uncertainty-analysis\limestone-demand-2023-2050-BAU-50km-M.tif").read(1)

total_demand = sand + gravel + limestone  # 单位: t (per pixel)
global_total = total_demand.sum() / 1e9   # Gt

In [53]:
# 3. 检查 20 个缺失国家
only_in_mine = ['ALA', 'ATA', 'ATF', 'BLM', 'BVT', 'GGY', 'HMD', 'IMN', 
                'JEY', 'MAF', 'SGS', 'SJM', 'UMI', 'VIR', 'XAD', 'XCL', 
                'XKO', 'XPI', 'XSP', 'ZNC']

results = []
for iso in only_in_mine:
    row = country_pseudo_values[country_pseudo_values['country'] == iso]
    if row.empty:
        results.append((iso, None, 0, '⚠️ 未找到'))
        continue
    code = row['value'].values[0]
    full_name = row['full name'].values[0]
    mask = (country_raster_50km == code)
    pixel_count = mask.sum()
    d_gt = total_demand[mask].sum() / 1e9
    d_pct = 100 * d_gt / global_total
    results.append((iso, full_name, pixel_count, d_gt, d_pct))

In [54]:
# 4. 排序展示
df = pd.DataFrame(results, columns=['ISO3', 'name', 'pixels', 'demand_Gt', 'pct_of_global'])
df = df.sort_values('demand_Gt', ascending=False)
print(df.to_string(index=False))

print(f"\n全球总需求 (sand+gravel+limestone, BAU-M): {global_total:.2f} Gt")
print(f"20 国合计: {df['demand_Gt'].sum():.4f} Gt ({100*df['demand_Gt'].sum()/global_total:.3f}%)")

ISO3                             name  pixels  demand_Gt  pct_of_global
 ALA                           ??land       1        0.0            0.0
 ATA                       Antarctica    4860        0.0            0.0
 ATF      French Southern Territories       4        0.0            0.0
 BLM                Saint-Barth??lemy       0        0.0            0.0
 BVT                    Bouvet Island       0        0.0            0.0
 GGY                         Guernsey       0        0.0            0.0
 HMD Heard Island and McDonald Island       1        0.0            0.0
 IMN                      Isle of Man       0        0.0            0.0
 JEY                           Jersey       0        0.0            0.0
 MAF                     Saint-Martin       0        0.0            0.0
 SGS South Georgia and the South Sand       2        0.0            0.0
 SJM           Svalbard and Jan Mayen      24        0.0            0.0
 UMI United States Minor Outlying Isl       0        0.0        